# LinkNet - Uydudan yol hasari segmentasyonu

**Katman:** Zorunlu  |  **Kodlayici:** `resnet18`  |  **Kod cozucu:** `Linknet`

U-Net'in artik bloklu varyanti - kodlayici etkisini izole eder.

Bizim yontem: **SegFormer (MiT-B4)**, focal+dice, ortusmeli karolama + morfolojik son-islem.  
Butun mimariler `seg_common.py` uzerinden AYNI veri / AYNI kayip / AYNI metrik.  
Metrik: IoU/mIoU, F1, Kesinlik, Duyarlilik, Ozgulluk, Dogruluk, AUC + Params(M)/GFLOPs/cikarim(ms).  
> M2'nin dersi: bu gorevde dogruluk tek basina aldaticidir; IoU ile birlikte verilir.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/01_yol_hasari
    %pip install -q segmentation-models-pytorch albumentations roboflow thop opencv-python-headless
print('cwd:', os.getcwd())

## 1. Veri seti (Roboflow Universe)

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import seg_common as SC

# === AYAR: kendi Roboflow bilgilerini gir (ya da seg_common.py > CONFIG) ===
SC.CONFIG.update(dict(
    source='roboflow',
    roboflow_api_key='BURAYA_ROBOFLOW_API_KEY',
    roboflow_workspace='workspace-slug',
    roboflow_project='project-slug',
    roboflow_version=1,
))
paths = SC.prepare_dataset()
dl = SC.build_loaders(paths)
print({k: len(v.dataset) for k, v in dl.items()})

## 2. Model

In [ ]:
MODEL_NAME = "linknet_resnet18"
ARCH, ENCODER = "Linknet", "resnet18"
PARAMS = dict(arch=ARCH, encoder=ENCODER, **{k: SC.CONFIG[k] for k in ['img_size','batch_size','epochs','lr']})
model = SC.build_model(ARCH, ENCODER, num_classes=SC.CONFIG['num_classes'])

## 3. Egitim (focal + dice, cosine LR)

In [ ]:
model = SC.train_model(model, dl, SC.CONFIG)

## 4. Test metrikleri + verimlilik

In [ ]:
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
test_split = 'test' if 'test' in dl else 'valid'
metrics = SC.evaluate_seg(model, dl[test_split], dev)
for k, v in metrics.items(): print(f'{k:12s}: {v:.4f}')

eff = SC.efficiency_stats(model, SC.CONFIG['img_size'], device='cpu')
if dev == 'cuda': eff.update(SC.efficiency_stats(model, SC.CONFIG['img_size'], device='cuda'))
print(eff)

## 5. Kaydet

In [ ]:
lb = SC.save_result(MODEL_NAME, metrics, eff, extra={'params': PARAMS})
lb[['model','mean_iou','f1','precision','recall','accuracy','params_m']]